# <font color="#00ced1"><b>Target Marketing for Canadian Bank</b></font>
Inclass Classification problem for AML-1413 Winter 2023: This leaderboard is calculated with approximately 40% of the test data. The final results will be based on the other 60%, so the final standings may be different.

>**Group Theta**
>- Erwin Mongui
>- Dev Makwana
>- Aparna Manjalavil
>- Yasar Mansuri
>- Amarnadh Movva

**Business Case**: Target Marketing for Canadian Bank

**Data & Industry**

In this competition, you will be working on real customer data from a major international bank and the provided data are real, but de-identified to protect both customer and bank privacy.  So, you will have part of (not all) the challenges of dealing with real data which is an invaluable experience and knowledge that will significantly improves your success chance in the area of data science.

**Scenario**
In order to increase deposit balances, Bank “A” is looking to execute a marketing campaign. The campaign will target existing clients and will offer them promotional interest rates to attract deposit balances.

1. [Business Case Evaluation](#1)
2. [Data Identification](#2)
   1. [Metadata](#2.1)
3. [Supervised learning](#3)
    1. [CatBoostClassifier](#3.1)
    2. [ComplementNB](#3.2)
    3. [XGBClassifier](#3.3)
    4. [GradientBoostingClassifier](#3.4)
    5. [GaussianNB](#3.5)
    6. [ExtraTreesClassifier](#3.6)
    7. [AdaBoostClassifier](#3.7)
    8. [Logistic Regression](#3.8)
    9. [Linear support vector machines](#3.9)
    10. [k-nearest neighbors](#3.10)
    11. [DecisionTreeClassifier](#3.11)
    12. [MLPClassifier](#3.12)
    13. [RandomForestClassifier](#3.13)
10. [Unsupervised learning](#5)


## Business Case Evaluation <a class="anchor" id="1"></a>
[Back to Table of Contents](#0.1)

You are tasked to find right set of customers who are most likely to respond to the campaign.

## Data Identification <a class="anchor" id="2"></a>
[Back to Table of Contents](#0.1)

**Datasets**
<br/>

**Training data** –Training dataset contains 64000 observations from previous campaigns. Actual responses (labels) are in the variable called ‘Target’.

**Testing data** – contains 1478 observations with 36 variables and no labels.

### Metadata <a class="anchor" id="2.1"></a>
[Back to Table of Contents](#0.1)

**File descriptions**
train.csv - the training set
test.csv - the test set

**Data fields**
<br/>

|COLUMN|DESCRIPTION|
|:--|:--|
|**customer_id**| an anonymous id unique to a given customer|
|**Balance**| Credit Card Balance|
|**PreviousCampaignResult**| Last Campaign Results|
|**Product1...Product6**| List of different products owned by the customer|
|**Transaction1...Transaction9**| Last 9 transactions made by the customer|
|**External Accounts 1...External Accounts 7**| External accounts held by the customer, including bills|
|**Activity Indicator**| Number of activities done the customer, including telebanking, going to the bank|
|**Regular Interaction Indicator**| How frequent the customer is interacting with the bank (Rating Scale)|
|**CompetitiveRate1 ... CompetitiveRate7**| Interest rates offered to the customer|
|**RateBefore**| Interest rates on the products, customer have before the competitive rates being offered|
|**ReferenceRate**|Interest rates on the products, the customer have after the negotiations made to the competitive rates being offered|

In [ ]:
from sklearn.metrics import roc_curve, auc
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import roc_auc_score
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import warnings
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix
from matplotlib.legend_handler import HandlerLine2D
from sklearn.ensemble import IsolationForest
from sklearn.cluster import DBSCAN
from scipy import stats
from typing import List, Tuple
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.naive_bayes import ComplementNB
from sklearn.cluster import KMeans
import xgboost as xgb
from sklearn.preprocessing import MinMaxScaler
from catboost import CatBoostClassifier

# Ignore Warnings numpy and pandas
np.seterr(divide="ignore", invalid="ignore")
warnings.filterwarnings("ignore")

# Display full columns
pd.set_option("display.max_columns", None)
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['xtick.labelsize'] = 12
plt.rcParams['ytick.labelsize'] = 12

Function aim is validate different hyperparameter in order to find the best parameter

In [ ]:
def validate_hyperparameter(model: object, hyperparameter: str, values: object) -> None:
    """
    Validate hyperparameter
    :param model: model object
    :param hyperparameter: parameter name
    :param values: range of values
    """
    train_results = []
    test_results = []

    for item in values:
        o_model = model(**{hyperparameter: item})
        o_model.fit(X_train, y_train)
        train_pred = o_model.predict(X_train)

        false_positive_rate, true_positive_rate, thresholds = roc_curve(y_train, train_pred)
        roc_auc = auc(false_positive_rate, true_positive_rate)
        train_results.append(roc_auc)
        y_pred = o_model.predict(X_test)

        false_positive_rate, true_positive_rate, thresholds = roc_curve(y_test, y_pred)
        roc_auc = auc(false_positive_rate, true_positive_rate)
        test_results.append(roc_auc)

    line_1, = plt.plot(values, train_results, 'b', label='Train AUC')
    for x, y in zip(values, train_results):
        plt.text(x=x, y=y, s="{} [{:0.3}]".format(x, y), color="purple")
    line_2, = plt.plot(values, test_results, 'r', label='Test AUC')
    for x, y in zip(values, test_results):
        plt.text(x=x, y=y, s="{} [{:0.3}]".format(x, y), color="purple")

    plt.legend(handler_map={line_1: HandlerLine2D(numpoints=2)})
    plt.legend(handler_map={line_2: HandlerLine2D(numpoints=2)})
    plt.title(model.__name__)
    plt.ylabel('AUC score')
    plt.xlabel('max features')
    plt.show()

Create a summary about the dataset to identify null and duplicated values.

In [ ]:
def get_summary(data: pd.DataFrame) -> pd.DataFrame:
    """
    Calculate overall dataset statistics
    :param data: initial dataframe
    :return: summary dataframe
    """

    summary = pd.DataFrame(
        {
            "Count": data.count(),
            "Difference": len(data) - data.count(),
            "%Difference": ((len(data) - data.count()) * 100) / len(data),
            "Unique": data.nunique(),
            "%Unique": data.nunique() * 100 / len(data),
            "Duplicated": data.duplicated().sum(),
            "%Duplicated": data.duplicated().sum() * 100 / len(data),
            "Null": data.isna().sum(),
            "%Null": data.isnull().sum() * 100 / len(data),
            "type": data.dtypes,
        }
    )
    return summary

Function created to transform the data in a set of groups.

In [ ]:
def create_calculated_fields(data: pd.DataFrame) -> None:
    """
    Create calculated fields to the model
    :param data: initial dataframe
    """
    # total number of products owned by the customer
    data["TotalProducts"] = data[
        ["Product1", "Product2", "Product3", "Product4", "Product5", "Product6"]
    ].sum(axis=1)

    # total negative transaction made by the customer
    data["CountNegativeTransactions"] = (
        data[
            [
                "Transaction1",
                "Transaction2",
                "Transaction3",
                "Transaction4",
                "Transaction5",
                "Transaction6",
                "Transaction7",
                "Transaction8",
                "Transaction9",
            ]
        ]
        .fillna(0)
        .lt(0)
        .sum(axis=1)
    )
    # total positive transaction made by the customer
    data["CountPositiveTransactions"] = (
        data[
            [
                "Transaction1",
                "Transaction2",
                "Transaction3",
                "Transaction4",
                "Transaction5",
                "Transaction6",
                "Transaction7",
                "Transaction8",
                "Transaction9",
            ]
        ]
        .fillna(0)
        .gt(0)
        .sum(axis=1)
    )

    data["CountTransactions"] = data[
        ["CountNegativeTransactions", "CountPositiveTransactions"]
    ].sum(axis=1)

    data["AverageTransactions"] = data[
        [
            "Transaction1",
            "Transaction2",
            "Transaction3",
            "Transaction4",
            "Transaction5",
            "Transaction6",
            "Transaction7",
            "Transaction8",
            "Transaction9",
        ]
    ].mean(axis=1)

    # average interest rate offered to the customer
    data["AverageInterestRate"] = data[
        [
            "CompetitiveRate1",
            "CompetitiveRate2",
            "CompetitiveRate3",
            "CompetitiveRate4",
            "CompetitiveRate5",
            "CompetitiveRate6",
            "CompetitiveRate7",
        ]
    ].mean(axis=1)

    # total number of external accounts owned by the customer
    data["TotalExternalAccount"] = data[
        [
            "ExternalAccount1",
            "ExternalAccount2",
            "ExternalAccount3",
            "ExternalAccount4",
            "ExternalAccount5",
            "ExternalAccount6",
            "ExternalAccount7",
        ]
    ].sum(axis=1)

    # Create Balance Groups depending on different ranges
    data["BalanceGroup1"] = np.where((data["Balance"] >= 10000) & (data["Balance"] < 20000), 1, 0)
    data["BalanceGroup2"] = np.where((data["Balance"] >= 20001) & (data["Balance"] < 40000), 1, 0)
    data["BalanceGroup3"] = np.where((data["Balance"] >= 40001) & (data["Balance"] <= 50000), 1, 0)

    # Total Interest related to balance
    data["TotalAverageInterestBalance"] = np.where(data["ReferenceRate"] > 0, data["Balance"] / data["ReferenceRate"], 0)

Function created to get the available metrics about the performance of the model. The metrics are Confusion Matrix, AUC Score, Accuracy, precision, recall, f1-score, and support.

In [ ]:
def get_metrics(text_model: object, true_value: object, pred_value: object) -> None:
    print(f"---{text_model}---")
    print(classification_report(true_value, pred_value))

    cm = confusion_matrix(true_value, pred_value)
    print("Confusion Matrix", cm)

    auc_score = roc_auc_score(true_value, pred_value)
    print("AUC Score:", auc_score)

Read train and test datasets

In [ ]:
df_train = pd.read_csv("../input/target-marketing-for-canadian-bank-2023w-aml1413/train.csv")
df_test = pd.read_csv("../input/target-marketing-for-canadian-bank-2023w-aml1413/test.csv")

Get the shape of train dataset

In [ ]:
print(f"TRAIN (SHAPE):\t {df_train.shape}")
print(f"TEST (SHAPE):\t {df_test.shape}")

Get basic dataframe information

In [ ]:
summary_train = get_summary(df_train)
summary_test = get_summary(df_test)

In [ ]:
summary_train

In [ ]:
summary_test

Correlation matrix to identify some dependencies.

In [ ]:
corr_mat = df_train.corr(method='spearman')
cg = sns.clustermap(corr_mat, cmap="YlGnBu", linewidths=0.1, figsize=(20, 20));
plt.setp(cg.ax_heatmap.yaxis.get_majorticklabels(), rotation=0)
cg
plt.show()

In [ ]:
# Create calculated fields
create_calculated_fields(df_test)
create_calculated_fields(df_train)

# Products column list
list_products = df_train.loc[
                :, df_train.columns.str.startswith("Product")
                ].columns.values

# Copy dataset
df_train_cleaned = df_train.copy()

# Drop customer id
train = df_train_cleaned.drop(
    columns=[
        "Customer_id"
    ]
)

# Get numeric columns from categorical
train = pd.get_dummies(
    train,
    columns=[
        "ActivityIndicator",
        "PreviousCampaignResult",
        "RegularInteractionIndicator",
    ],
)
test = pd.get_dummies(
    df_test,
    columns=[
        "ActivityIndicator",
        "PreviousCampaignResult",
        "RegularInteractionIndicator",
    ],
)

# Normalize the data
scaler_columns = [
    "Balance",
    "Transaction1",
    "Transaction2",
    "Transaction3",
    "Transaction4",
    "Transaction5",
    "Transaction6",
    "Transaction7",
    "Transaction8",
    "Transaction9",
    "CompetitiveRate1",
    "CompetitiveRate2",
    "CompetitiveRate3",
    "CompetitiveRate4",
    "CompetitiveRate5",
    "CompetitiveRate6",
    "CompetitiveRate7",
    "RateBefore",
    "ReferenceRate",
]
scaler = StandardScaler()
train[scaler_columns] = scaler.fit_transform(train[scaler_columns])
scaler_test = StandardScaler()
test[scaler_columns] = scaler_test.fit_transform(test[scaler_columns])

# Delete duplicated values
train.drop_duplicates(inplace=True)

# Supervised learning <a class="anchor" id="3"></a>
[Back to Table of Contents](#0.1)

Splitting of data as test and training set using Python (75:25 or 80:20)

In [ ]:
X_train_md = train.drop(["Target"], axis=1)
y_train_md = train["Target"]
X_test_md = test.loc[:, test.columns.isin(X_train_md.columns)]

X = X_train_md
y = y_train_md

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

## CatBoostClassifier <a class="anchor" id="3.1"></a>
[Back to Table of Contents](#0.1)

CatBoostClassifier model that was trained to predict a binary classification problem. Based on the evaluation results, the model has an accuracy of 0.63, which means it correctly predicted the target classes for 63% of the instances in the test dataset. The precision and recall scores for both classes are also similar to each other and to those of the Gradient Boosting Classifier model, with a precision of 0.62 for class 0 and 0.64 for class 1, and a recall of 0.67 for class 0 and 0.59 for class 1.

The AUC score of 0.629 indicates that the model performs slightly better than random chance at distinguishing between the two classes. The confusion matrix shows that the model correctly predicted 5334 instances of class 0 and 4733 instances of class 1, while misclassifying 2670 instances of class 0 and 3263 instances of class 1.

In [ ]:
params = {
    'iterations': 498,
    'learning_rate': 0.08,
    'depth': 4,
    'loss_function': 'Logloss',
    'verbose': False,
    'subsample': 1,
}

cbc = CatBoostClassifier(**params)

cbc.fit(X_train, y_train)
cbc_y_pred = cbc.predict(X_test)
get_metrics("CatBoostClassifier", y_test, cbc_y_pred)

## ComplementNB <a class="anchor" id="3.2"></a>
[Back to Table of Contents](#0.1)

Complement Naive Bayes Classifier model that was trained to predict a binary classification problem. Based on the evaluation results, the model has an accuracy of 0.59, which means it correctly predicted the target classes for 59% of the instances in the test dataset. The precision and recall scores for both classes are also similar to each other, with a precision of 0.58 for class 0 and 0.61 for class 1, and a recall of 0.66 for class 0 and 0.53 for class 1.

The AUC score of 0.592 indicates that the model performs slightly better than random chance at distinguishing between the two classes. The confusion matrix shows that the model correctly predicted 5254 instances of class 0 and 4221 instances of class 1, while misclassifying 2750 instances of class 0 and 3775 instances of class 1.

In [ ]:
scaler = MinMaxScaler()
cnb = ComplementNB(alpha=6.0)

cnb.fit(scaler.fit_transform(X_train), y_train)
cnb_y_pred = cnb.predict(scaler.transform(X_test))
get_metrics("ComplementNB", y_test, cnb_y_pred)

## XGBClassifier <a class="anchor" id="3.3"></a>
[Back to Table of Contents](#0.1)

The evaluation metrics provided include precision, recall, f1-score, and support for each class, as well as the accuracy, macro-averaged f1-score, weighted-averaged f1-score, confusion matrix, and AUC score. The precision, recall, and f1-score for both classes (0 and 1) are similar, with slightly higher values for class 1. This indicates that the model is performing similarly well for both classes in terms of identifying true positives and true negatives.

The accuracy of the model is 0.63, which means that the model correctly predicted the target class for 63% of the instances in the test set. The macro-averaged f1-score is 0.63, which is the average of the f1-score for each class, weighted equally. The weighted average of the f1-score is also 0.63, which takes into account the number of samples in each class.

The confusion matrix shows that the model correctly predicted 5314 instances of class 0 and 4717 instances of class 1, but misclassified 2690 instances of class 0 and 3279 instances of class 1. The AUC score is 0.626919, which is a measure of the model's ability to distinguish between the two classes. A score of 0.5 indicates that the model is randomly guessing, while a score of 1.0 indicates perfect classification.

In [ ]:
params = {
    'learning_rate': 0.08,
    'subsample': 1,
}

xgb_m = xgb.XGBClassifier(**params)

xgb_m.fit(X_train, y_train)
xgb_m_y_pred = xgb_m.predict(X_test)
get_metrics("XGBClassifier", y_test, xgb_m_y_pred)

## GradientBoostingClassifier <a class="anchor" id="3.4"></a>
[Back to Table of Contents](#0.1)

The result indicates how well the model performs in predicting two classes (0 and 1) based on the given metrics: precision, recall, f1-score, support, accuracy, and AUC score.

Precision indicates the proportion of positive predictions that are correctly classified, while recall is the proportion of actual positives that are correctly classified by the model. F1-score is the weighted harmonic mean of precision and recall, and it provides a single score that combines both metrics. Support is the number of actual occurrences of the class in the specified dataset.

The accuracy of the model is 63%, which means it correctly classified 63% of the test data. The weighted average of precision and recall for class 0 is 0.62, which means the model predicted 62% of the total true positives for class 0. Similarly, the weighted average of precision and recall for class 1 is 0.64, indicating the model predicted 64% of the total true positives for class 1.

The confusion matrix represents the number of true positives, false positives, true negatives, and false negatives for each class. For class 0, the model correctly predicted 5322 true positives and 2682 false negatives, while for class 1, it predicted 4740 true positives and 3256 false negatives.

The AUC score is 0.63, which is an indicator of how well the model is capable of distinguishing between positive and negative classes. In general, an AUC score of 0.5 indicates that the model has no discrimination capacity, and an AUC score of 1 indicates a perfect classification model.

In [ ]:
# params = {
#     'n_estimators': 498,
#     'learning_rate': 0.08,
#     'max_depth': 4,
#     'verbose': False,
#     'subsample': 1,
# }

gbr = GradientBoostingClassifier(random_state=42)

gbr.fit(X_train, y_train)
gbr_y_pred = gbr.predict(X_test)
get_metrics("GradientBoostingClassifier", y_test, gbr_y_pred)

### GradientBoostingClassifier :: Hyperparameter :: learning_rate

In [ ]:
validate_hyperparameter(GradientBoostingClassifier, "learning_rate", [0.08, 0.1, 0.25, 0.5, 1])

### GradientBoostingClassifier :: Hyperparameter :: N_estimators

In [ ]:
validate_hyperparameter(GradientBoostingClassifier, "n_estimators", [32, 64, 100, 200, 498])

### GradientBoostingClassifier :: Hyperparameter :: max_depth

In [ ]:
validate_hyperparameter(GradientBoostingClassifier, "max_depth", [1, 3, 4, 9])

### GradientBoostingClassifier :: Hyperparameter :: min_samples_split

In [ ]:
validate_hyperparameter(GradientBoostingClassifier, "min_samples_split", np.linspace(0.1, 1.0, 10, endpoint=True))

### GradientBoostingClassifier :: Hyperparameter :: min_samples_leaf

In [ ]:
validate_hyperparameter(GradientBoostingClassifier, "min_samples_leaf", np.linspace(0.1, 0.5, 5, endpoint=True))

### GradientBoostingClassifier :: Hyperparameter :: max_features

In [ ]:
validate_hyperparameter(GradientBoostingClassifier, "max_features", [40, 42, 45, 47, 48, 50])

## GaussianNB <a class="anchor" id="3.5"></a>
[Back to Table of Contents](#0.1)

The model has been trained on a dataset with 16000 instances and two target classes (0 and 1). The evaluation results are as follows:

1) **Precision**: The proportion of true positive predictions (TP) over all positive predictions (TP + false positive predictions (FP)). For class 0, the precision is 0.52 and for class 1, the precision is 0.66.
2) **Recall**: The proportion of true positive predictions (TP) over all actual positive instances (TP + false negative predictions (FN)). For class 0, the recall is 0.92 and for class 1, the recall is 0.15.
3) **F1-score**: The harmonic mean of precision and recall, which provides a balanced measure of the two metrics. For class 0, the F1-score is 0.67 and for class 1, the F1-score is 0.24.
4) **Support**: The number of instances in each class in the test dataset. For class 0, there are 8004 instances and for class 1, there are 7996 instances.
5) **Accuracy**: The proportion of correct predictions over all predictions. The model achieves an overall accuracy of 0.54.
6) **Macro-average**: The average of precision, recall, and F1-score for both classes. The macro-average precision, recall, and F1-score are 0.59, 0.54, and 0.46, respectively.
7) **Weighted-average**: The average of precision, recall, and F1-score weighted by the number of instances in each class. The weighted-average precision, recall, and F1-score are all 0.59.
8) **Confusion Matrix**: A table showing the actual and predicted class labels. In this case, the confusion matrix shows that 7401 instances of class 0 were correctly predicted, 603 instances of class 0 were incorrectly predicted as class 1, 6803 instances of class 1 were incorrectly predicted as class 0, and 1193 instances of class 1 were correctly predicted.
9) **AUC Score**: A metric that measures the overall performance of the model in distinguishing between the two classes, taking into account both true positive rate and false positive rate. In this case, the AUC Score is 0.5369, which indicates that the model's performance is only slightly better than random chance (0.5) and may need further improvement.

In [ ]:
gnb = GaussianNB()

gnb.fit(X_train, y_train)
gnb_y_pred = gnb.predict(X_test)
get_metrics("GaussianNB", y_test, gnb_y_pred)

## ExtraTreesClassifier <a class="anchor" id="3.6"></a>
[Back to Table of Contents](#0.1)

These are the evaluation metrics of a model trained using the ExtraTreesClassifier algorithm on a binary classification problem. Here is what each metric means:

Precision: the proportion of true positive predictions among all positive predictions. In this case, the precision for the positive class (1) is 0.57, which means that when the model predicts a sample as positive, it is correct 57% of the time.

Recall: the proportion of true positive predictions among all actual positive samples. In this case, the recall for the positive class (1) is 0.62, which means that the model correctly identifies 62% of all positive samples.

F1-score: the harmonic mean of precision and recall, which provides a single score that balances both metrics. In this case, the F1-score for the positive class (1) is 0.59.

Support: the number of samples in each class.

Accuracy: the proportion of correct predictions among all samples. In this case, the accuracy is 0.58, which means that the model correctly predicts the class for 58% of all samples.

Confusion matrix: a table that shows the number of true positives, true negatives, false positives, and false negatives for each class.

AUC score: the area under the receiver operating characteristic curve, which measures the trade-off between true positive rate and false positive rate for different classification thresholds. In this case, the AUC score is 0.577, which means that the model has a moderate ability to distinguish between positive and negative samples.

In [ ]:
etc = ExtraTreesClassifier(random_state=42)
etc.fit(X_train, y_train)
etc_y_pred = etc.predict(X_test)
get_metrics("ExtraTreesClassifier", y_test, etc_y_pred)

## AdaBoostClassifier <a class="anchor" id="3.7"></a>
[Back to Table of Contents](#0.1)

The AdaBoostClassifier is a boosting algorithm that creates a strong classifier from a number of weak classifiers. The result shows the performance of the model on a dataset, where it achieved an accuracy of 0.62. The precision, recall, and f1-score for the model are 0.61, 0.66, and 0.64, respectively, for class 0, and 0.63, 0.58, and 0.61, respectively, for class 1. The confusion matrix shows the count of true negatives, false positives, false negatives, and true positives, respectively, as [5310, 2694, 3350, 4646]. The AUC score is 0.622, which means that the model has a moderate performance in distinguishing between positive and negative classes.

In [ ]:
adab = AdaBoostClassifier(random_state=0)
adab.fit(X_train, y_train)
adab_y_pred = adab.predict(X_test)
get_metrics("AdaBoostClassifier", y_test, adab_y_pred)

## Logistic Regression <a class="anchor" id="3.8"></a>
[Back to Table of Contents](#0.1)

The precision, recall, and F1-score are provided for both classes (0 and 1), along with the support (number of samples) for each class. The accuracy of the model is also given.

In this case, the precision for class 0 is 0.61, meaning that out of all the samples predicted as class 0, 61% were actually class 0. The recall for class 0 is 0.65, meaning that out of all the actual class 0 samples, 65% were correctly predicted as class 0. The F1-score is the harmonic mean of precision and recall for each class. The support for class 0 is 8004, meaning there are 8004 samples in the test set that are actually class 0.

Similarly, for class 1, the precision is 0.62, meaning that out of all the samples predicted as class 1, 62% were actually class 1. The recall for class 1 is 0.58, meaning that out of all the actual class 1 samples, 58% were correctly predicted as class 1. The F1-score is the harmonic mean of precision and recall for each class. The support for class 1 is 7996, meaning there are 7996 samples in the test set that are actually class 1.

The confusion matrix is also provided, which shows the number of true positives, true negatives, false positives, and false negatives for each class. In this case, there are 5186 true negatives (predicted as 0 and actually 0), 2818 false positives (predicted as 0 but actually 1), 3346 false negatives (predicted as 1 but actually 0), and 4650 true positives (predicted as 1 and actually 1).

Finally, the AUC score is provided, which is the area under the receiver operating characteristic (ROC) curve. The AUC score measures the performance of the model across all possible classification thresholds. In this case, the AUC score is 0.6147, indicating that the model's ability to distinguish between the two classes is only slightly better than random guessing (AUC of 0.5).

In [ ]:
validate_hyperparameter(LogisticRegression, "C", [0.01, 0.1, 1.0])

In [ ]:
validate_hyperparameter(LogisticRegression, "max_iter", [50, 100, 200, 300])

In [ ]:
validate_hyperparameter(LogisticRegression, "solver", ['newton-cg', 'liblinear', 'lbfgs'])

In [ ]:
logr = LogisticRegression(random_state=42, solver='newton-cg')
logr.fit(X_train, y_train)
logr_y_pred = logr.predict(X_test)
get_metrics("LogisticRegression", y_test, logr_y_pred)

## Linear support vector machines <a class="anchor" id="3.9"></a>
[Back to Table of Contents](#0.1)

Linear Support Vector Classifier (LinearSVC) algorithm, which is a type of binary classification algorithm:

The precision of class 0 (the first class) is 0.51, meaning that when the model predicts the first class, it is correct only 51% of the time. The recall of class 0 is 0.97, meaning that out of all the instances that actually belong to class 0, the model correctly identified 97% of them. The f1-score of class 0 is 0.67, which is the harmonic mean of precision and recall.

Similarly, the precision, recall, and f1-score of class 1 (the second class) are 0.71, 0.08, and 0.15 respectively.

The accuracy of the model is 0.52, which means that the model correctly classified 52% of the instances in the test set.

The confusion matrix shows the number of true positives, false positives, true negatives, and false negatives. In this case, the model correctly classified 7735 instances of class 0 and incorrectly classified 269 instances of class 0 as class 1. The model correctly classified only 656 instances of class 1 and incorrectly classified 7340 instances of class 1 as class 0.

The AUC score (Area Under the ROC Curve) is 0.5242, which is a metric used to evaluate binary classification algorithms that takes into account both the true positive rate and the false positive rate.

In [ ]:
clf = LinearSVC(random_state=42, C=4)
clf.fit(X_train, y_train)
clf_y_pred = clf.predict(X_test)

get_metrics("LinearSVC", y_test, clf_y_pred)

## k-nearest neighbors <a class="anchor" id="3.10"></a>
[Back to Table of Contents](#0.1)

The results of the model indicate the performance of the KNeighborsClassifier algorithm on a dataset.

The precision, recall, and F1-score are the metrics used to evaluate the performance of the model. Precision is the fraction of true positives over the total number of true and false positives, recall is the fraction of true positives over the total number of true positives and false negatives, and F1-score is the harmonic mean of precision and recall.

The support column indicates the number of samples in each class in the dataset. The accuracy score is the proportion of correctly classified samples out of the total number of samples.

The confusion matrix shows the number of true positives, true negatives, false positives, and false negatives.

Finally, the AUC score is the area under the ROC curve, which is a plot of the true positive rate versus the false positive rate.

In this case, the KNeighborsClassifier model achieved an accuracy of 0.56, with similar precision, recall, and F1-score for both classes. The AUC score is also close to 0.5, which indicates that the model's performance is not significantly better than random guessing. Therefore, the model's performance can be improved with better feature engineering, parameter tuning, or using other algorithms.

In [ ]:
knn = KNeighborsClassifier(p=4, leaf_size=1, n_neighbors=5)
knn.fit(X_train, y_train)
knn_y_pred = knn.predict(X_test)

get_metrics("KNeighborsClassifier", y_test, knn_y_pred)

## DecisionTreeClassifier <a class="anchor" id="3.11"></a>
[Back to Table of Contents](#0.1)

The model is a decision tree classifier, and the results are showing the performance of the model.

The precision, recall, and F1-score are measures of the model's accuracy. Precision is the ratio of true positive (TP) to TP plus false positive (FP) predictions. Recall is the ratio of TP to TP plus false negative (FN) predictions. F1-score is the harmonic mean of precision and recall.

In this case, the precision and recall are almost the same, which indicates that the model is not biased towards one class. The F1-score is also similar for both classes.

The accuracy is the proportion of correct predictions among all the predictions, and it is 0.56, which means that the model correctly predicted 56% of the cases.

The confusion matrix shows the number of true negatives (TN), false positives (FP), false negatives (FN), and true positives (TP) predictions made by the model. In this case, the model made 4620 TN, 3384 FP, 3603 FN, and 4393 TP predictions.

The AUC score is a measure of the model's ability to distinguish between positive and negative classes. An AUC score of 0.563 indicates that the model is not performing well in differentiating between the two classes.

In [ ]:
dcst = DecisionTreeClassifier(random_state=0, min_samples_leaf=9)
dcst.fit(X_train, y_train)
dcst_y_pred = dcst.predict(X_test)

get_metrics("DecisionTreeClassifier", y_test, dcst_y_pred)

## MLPClassifier <a class="anchor" id="3.12"></a>
[Back to Table of Contents](#0.1)

The model results show the performance metrics of the MLPClassifier algorithm. It is a neural network-based algorithm that is used for classification problems. The performance metrics in the result indicate that the algorithm did not perform well on the given dataset.

The precision score for class 0 is 0.50 which means out of all predicted 0s, 50% are true 0s. The recall score for class 0 is 1.00 which means out of all actual 0s, the model is able to identify all of them. The f1-score is the harmonic mean of precision and recall, and for class 0, it is 0.67. The support for class 0 is 8004 which is the number of actual 0s in the dataset.

For class 1, the precision score is 0.00 which means out of all predicted 1s, none are true 1s. The recall score is 0.00 which means out of all actual 1s, the model is not able to identify any of them. The f1-score is 0.00 and the support for class 1 is 7996.

The accuracy of the model is 0.50 which means it predicted only half of the data correctly. The macro-average of precision, recall, and f1-score is 0.25 for both classes and the weighted average is also 0.25.

The confusion matrix shows that the model predicted all data points as 0s for class 1, which means it did not detect any of the positive class instances. The AUC score is 0.5 which is the same as random guessing and indicates that the model is not able to distinguish between the two classes.

In [ ]:
mlp = MLPClassifier(
    solver="lbfgs", alpha=1e-5, hidden_layer_sizes=(5, 2), random_state=42
)
mlp.fit(X_train, y_train)
mlp_y_pred = mlp.predict(X_test)

get_metrics("MLPClassifier", y_test, mlp_y_pred)

## RandomForestClassifier <a class="anchor" id="3.13"></a>
[Back to Table of Contents](#0.1)

Random Forest Classifier model that was trained to predict a binary classification problem. Based on the evaluation results, the model has an accuracy of 0.62, which means it correctly predicted the target classes for 62% of the instances in the test dataset. The precision and recall scores for both classes are also similar to each other and to those of the Gradient Boosting Classifier and CatBoostClassifier models, with a precision of 0.62 for class 0 and 0.63 for class 1, and a recall of 0.64 for class 0 and 0.61 for class 1.

The AUC score of 0.624 indicates that the model performs slightly better than random chance at distinguishing between the two classes. The confusion matrix shows that the model correctly predicted 5138 instances of class 0 and 4851 instances of class 1, while misclassifying 2866 instances of class 0 and 3145 instances of class 1.

In [ ]:
rfc = RandomForestClassifier(random_state=42, n_estimators=100, max_depth=20)
rfc.fit(X_train, y_train)
rfc_y_pred = rfc.predict(X_test)

get_metrics("RandomForestClassifier", y_test, rfc_y_pred)

Concat result get with the test dataframe

In [ ]:
columns_train = X_test.columns.values.tolist()
for column in columns_train:
    if column not in X_test_md.columns:
        X_test_md[column] = 0

In [ ]:
X_test_md = X_test_md[columns_train]
y_pred = gbr.predict(X_test_md)
result = pd.DataFrame(y_pred, columns=["Target"])
df_test_target = pd.concat([result, df_test], axis=1)
test = df_test_target[["Customer_id", "Target"]]

test.to_csv("Attempt37.csv", index=0)

## Unsupervised learning <a class="anchor" id="4"></a>
[Back to Table of Contents](#0.1)


In [ ]:
# Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

# Apply K-means algorithm with different numbers of clusters
n_clusters_list = [2, 3, 4, 5]

for n_clusters in n_clusters_list:
    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    y_pred = kmeans.fit_predict(X_train_scaled)

    plt.scatter(X_train_scaled[:, 0], X_train_scaled[:, 1], c=y_pred, s=50, cmap='viridis')
    plt.title(f"K-means clustering with {n_clusters} clusters")
    centers = kmeans.cluster_centers_
    plt.scatter(centers[:, 0], centers[:, 1], c='black', s=200, alpha=0.5)
    plt.show()

In [ ]:
# Select relevant features and scale them
X_test_scaled = scaler.transform(X_test)

# Predict the cluster for each data point in the testing set
for n_clusters in n_clusters_list:
    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    y_pred = kmeans.fit_predict(X_test_scaled)

    plt.scatter(X_test_scaled[:, 0], X_test_scaled[:, 1], c=y_pred, s=50, cmap='viridis')
    plt.title(f"K-means clustering with {n_clusters} clusters")
    centers = kmeans.cluster_centers_
    plt.scatter(centers[:, 0], centers[:, 1], c='black', s=200, alpha=0.5)
    plt.show()